# Demo Minesweeper (Colab)

Runs `Demo_test_Minesweeper.py` on Colab: launches the Flask Minesweeper server, opens a headless Chrome against it, and lets the trained agent play.

Make sure **Runtime -> Change runtime type -> GPU (T4 / L4 / A100)** is selected.

Open `TB.ipynb` in a separate tab to view TensorBoard. Independent kernel, will not block the demo.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === EDIT HERE if you renamed the Drive folder ===
DRIVE_ROOT_NAME = 'v3'
DRIVE_ROOT = f'/content/drive/MyDrive/{DRIVE_ROOT_NAME}'

%cd $DRIVE_ROOT
!ls

## 2. Install PyTorch 2.5.1 + CUDA 11.8
Match the local conda env `py310_torch251_cuda118`. Colab's preinstalled torch is usually cu12x; switching to cu118 keeps behavior aligned with local runs.

If the kernel already imported the old torch before this cell runs, do **Runtime -> Restart session** after install. A cold first run does not need a restart.

In [ ]:
!pip install -q --index-url https://download.pytorch.org/whl/cu118 \
    torch==2.5.1+cu118 torchvision==0.20.1+cu118

## 3. Install Google Chrome + MS fonts
- **Google Chrome** via official `.deb` (`chromium-browser` on modern Debian/Ubuntu is a snap wrapper that doesn't run in Colab).
- **`ttf-mscorefonts-installer`** ships real *Trebuchet MS* / *Courier New* — `static/style.css` uses those, and on Colab without them Chrome falls back to DejaVu/Liberation. That changes pixel widths and breaks the agent's template matching, not just the look.

chromedriver is fetched at runtime by `webdriver-manager` (in `util/Chrome_Driver.py`) so we don't apt-install it here.

No Xvfb needed — the X-dependent helpers (`pyautogui`, `pynput`) are bypassed via env vars in step 8.

In [ ]:
!apt-get update -qq

# Google Chrome
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb
!apt-get install -y -qq /tmp/chrome.deb
!rm -f /tmp/chrome.deb

# MS core fonts (Trebuchet MS / Courier New / Arial / ...) — pre-accept EULA so
# the install runs non-interactively, then refresh fontconfig cache so Chrome
# can find them at next launch.
!echo "ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true" | debconf-set-selections
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ttf-mscorefonts-installer
!fc-cache -f

# Sanity check — should list Trebuchet MS and Courier New
!fc-list | grep -iE 'trebuchet|courier new' | head -5

## 4. Install Python deps for the demo
Only the demo's actual call sites. `pyautogui` / `pynput` are intentionally **omitted** — their imports are guarded so X-less Colab doesn't load them.

In [ ]:
!pip install -q selenium webdriver-manager flask ultralytics

## 5. Environment check
Expected: `PyTorch : 2.5.1+cu118`, `CUDA build : 11.8`, `CUDA avail : True`, Chrome + chromedriver present.

In [ ]:
!nvidia-smi

import sys
import torch

print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA build  : {torch.version.cuda}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device      : {torch.cuda.get_device_name(0)}')
!google-chrome --version 2>/dev/null || chromium-browser --version 2>/dev/null || echo '[FAIL] no chrome — step 3 install did not work'
!echo '(chromedriver is fetched at runtime by webdriver-manager; not expected on PATH here)'

## 6. Anti-idle (reduce disconnects)
Colab disconnects after ~90 minutes of idle. Open browser **F12 -> Console** and paste this snippet; it simulates a click on the connect button every 60 seconds:

```javascript
function ClickConnect() {
    document.querySelector('colab-connect-button')
        ?.shadowRoot?.querySelector('#connect')?.click();
    console.log('anti-idle:', new Date().toLocaleTimeString());
}
setInterval(ClickConnect, 60000);
```

This does NOT bypass Google's hard limits (free tier ~12h/day, GPU quota). It only reduces idle-based disconnects.

## 7. Set up local workspace and background sync
Same FUSE-avoidance strategy as the training notebook: symlink read-only source, copy writeable models to local SSD, rsync back every 3 minutes.

Extra for demo:
- Also symlink `user_change/` (Game_Env's `rglob('user_change')` needs to find it under cwd).
- Also sync `testreport/` and `user_change/game_pic/` back (HTML reports and captured screenshots are written there).

In [ ]:
import os
import subprocess

WORKSPACE = '/content/workspace'
# DRIVE_ROOT defined in the "Mount Google Drive" section

os.makedirs(WORKSPACE, exist_ok=True)

# Symlink source tree (read-only from the demo's perspective).
code_link = f'{WORKSPACE}/autoTest_pytorch'
if not os.path.exists(code_link):
    os.symlink(f'{DRIVE_ROOT}/autoTest_pytorch', code_link)
    print(f'symlink: {code_link} -> {DRIVE_ROOT}/autoTest_pytorch')
else:
    print(f'(symlink already exists: {code_link})')

# Symlink user_change (Game_Env uses rglob('user_change') from cwd).
uc_link = f'{WORKSPACE}/user_change'
if not os.path.exists(uc_link):
    os.symlink(f'{DRIVE_ROOT}/user_change', uc_link)
    print(f'symlink: {uc_link} -> {DRIVE_ROOT}/user_change')
else:
    print(f'(symlink already exists: {uc_link})')

# Copy models from Drive to local SSD (demo keeps training, so writeable).
# To force a fresh copy, manually `!rm -rf /content/workspace/models` first.
local_models = f'{WORKSPACE}/models'
if not os.path.exists(local_models):
    print('Initial copy: Drive/models -> /content/workspace/models ...')
    subprocess.run(['cp', '-r', f'{DRIVE_ROOT}/models', local_models], check=True)
    print('done')
else:
    print(f'(reusing existing {local_models})')

# testreport directory (demo writes ./testreport/Report-*.html from cwd).
os.makedirs(f'{WORKSPACE}/testreport', exist_ok=True)

print(f'\nWorkspace ready at {WORKSPACE}')
!ls -la /content/workspace

In [ ]:
import os
import subprocess

# Background rsync: local workspace -> Drive every SYNC_INTERVAL seconds.
# No --delete, so an empty/partial local will not wipe Drive.
SYNC_INTERVAL = 180
PID_FILE = '/content/sync.pid'
LOG_FILE = '/content/sync.log'

if os.path.exists(PID_FILE):
    try:
        old_pid = int(open(PID_FILE).read().strip())
        subprocess.run(['kill', str(old_pid)], check=False)
        print(f'killed previous sync PID={old_pid}')
    except (ValueError, FileNotFoundError):
        pass

sync_cmd = f'''
while true; do
  sleep {SYNC_INTERVAL}
  rsync -a /content/workspace/models/      {DRIVE_ROOT}/models/      2>/dev/null
  rsync -a /content/workspace/testreport/  {DRIVE_ROOT}/testreport/  2>/dev/null
  rsync -a /content/workspace/user_change/game_pic/ {DRIVE_ROOT}/user_change/game_pic/ 2>/dev/null
  echo "[$(date +%H:%M:%S)] synced local -> Drive"
done
'''

sync_proc = subprocess.Popen(
    ['bash', '-c', sync_cmd],
    stdout=open(LOG_FILE, 'a'),
    stderr=subprocess.STDOUT,
)
with open(PID_FILE, 'w') as f:
    f.write(str(sync_proc.pid))

print(f'Background rsync running, PID={sync_proc.pid}')
print(f'  interval : every {SYNC_INTERVAL}s')
print(f'  log      : !tail -f {LOG_FILE}')
print(f'  stop     : !kill {sync_proc.pid}')

## 8. Set X-less env vars
Colab has no X display. Two env vars tell the demo to skip the X-dependent paths:

- **`AUTOTEST_HEADLESS_VIEWPORT=1920x1080`** — overrides `pyautogui.size()` in `util/Chrome_Driver.py`, so headless Chrome's CDP viewport is set without touching pyautogui.
- **`AUTOTEST_NO_KEYBOARD=1`** — skips the `pynput` keyboard listener in `Demo_test_Minesweeper.py`. The listener only exists to toggle pause on `End`, which Colab can't deliver anyway.

Set them in the kernel so the `!python` subprocess inherits them.

In [ ]:
import os

os.environ['AUTOTEST_HEADLESS_VIEWPORT'] = '1920x1080'
os.environ['AUTOTEST_NO_KEYBOARD'] = '1'

print(f'AUTOTEST_HEADLESS_VIEWPORT = {os.environ["AUTOTEST_HEADLESS_VIEWPORT"]}')
print(f'AUTOTEST_NO_KEYBOARD       = {os.environ["AUTOTEST_NO_KEYBOARD"]}')

## 9. Start the Minesweeper Flask server (background)
`Demo_test_Minesweeper.py` connects to `http://127.0.0.1:8000` (hard-coded in `util/Chrome_Driver.py`). The cell below launches `Minesweeper_web/server.py` in the background and waits until the port is responding before returning.

Re-running the cell kills any previous instance so the port is free.

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error

SERVER_PID_FILE = '/content/server.pid'
SERVER_LOG_FILE = '/content/server.log'
SERVER_URL = 'http://127.0.0.1:8000'
SERVER_SCRIPT = '/content/workspace/autoTest_pytorch/Minesweeper_web/server.py'

if os.path.exists(SERVER_PID_FILE):
    try:
        old_pid = int(open(SERVER_PID_FILE).read().strip())
        subprocess.run(['kill', str(old_pid)], check=False)
        print(f'killed previous server PID={old_pid}')
        time.sleep(1)
    except (ValueError, FileNotFoundError):
        pass

# Run from /content/workspace so relative paths inside the server resolve
# the same way as during a local run from the project root.
server_proc = subprocess.Popen(
    ['python', SERVER_SCRIPT],
    cwd='/content/workspace',
    stdout=open(SERVER_LOG_FILE, 'w'),
    stderr=subprocess.STDOUT,
)
with open(SERVER_PID_FILE, 'w') as f:
    f.write(str(server_proc.pid))

# Healthcheck: poll the port until it responds (or timeout).
TIMEOUT = 30
deadline = time.time() + TIMEOUT
ready = False
while time.time() < deadline:
    if server_proc.poll() is not None:
        print(f'[ERROR] server exited early (code={server_proc.returncode}). Last log:')
        !tail -n 30 {SERVER_LOG_FILE}
        raise RuntimeError('Minesweeper server failed to start')
    try:
        with urllib.request.urlopen(SERVER_URL, timeout=1) as resp:
            if resp.status < 500:
                ready = True
                break
    except (urllib.error.URLError, ConnectionResetError):
        time.sleep(0.5)

if not ready:
    print(f'[ERROR] server did not become ready within {TIMEOUT}s. Last log:')
    !tail -n 30 {SERVER_LOG_FILE}
    raise RuntimeError('Minesweeper server healthcheck failed')

print(f'Minesweeper server ready at {SERVER_URL}, PID={server_proc.pid}')
print(f'  log  : !tail -f {SERVER_LOG_FILE}')
print(f'  stop : !kill {server_proc.pid}')

## 10. Run the demo
Launch from the workspace, not from Drive. The demo writes to relative paths (`./models/`, `./testreport/`, `./user_change/game_pic/...`), so once cwd is the workspace it writes to local SSD.

Interrupting is fine (stop button or Runtime -> Interrupt). After interrupting, **always run the final-sync cell below**.

In [ ]:
%cd /content/workspace
!python autoTest_pytorch/Demo_test_Minesweeper.py

## 11. Final sync + shutdown (MUST run after interrupt or completion)
Background rsync only fires every 3 minutes, so the last chunk of progress is still on local SSD. Also stops the server and rsync so a re-run of the notebook starts clean. **Only close Colab after you see `final sync done`**.

In [ ]:
!rsync -av /content/workspace/models/      {DRIVE_ROOT}/models/      2>/dev/null
!rsync -av /content/workspace/testreport/  {DRIVE_ROOT}/testreport/  2>/dev/null
!rsync -av /content/workspace/user_change/game_pic/ {DRIVE_ROOT}/user_change/game_pic/ 2>/dev/null

import os, subprocess
for pid_file in ['/content/server.pid', '/content/sync.pid']:
    if os.path.exists(pid_file):
        try:
            pid = int(open(pid_file).read().strip())
            subprocess.run(['kill', str(pid)], check=False)
            print(f'killed PID={pid} ({pid_file})')
        except (ValueError, FileNotFoundError):
            pass

print('=== final sync done ===')